# 01 – Exploratory Data Analysis and Preprocessing

**Project:** Predictive Modeling for Drug Discovery via Virtual Screening  
**Student:** Milica Jeftić (ID: 89211255)  
**Date:** January 2026  
**Dataset:** Kaggle – Drug Discovery Virtual Screening Dataset

---

## Goal of This Notebook

This notebook performs exploratory data analysis (EDA) and preprocessing on the raw dataset loaded in notebook 00.
The objectives are:

1. **Environment Setup** – Import libraries and load raw data from notebook 00
2. **Exploratory Data Analysis** – Examine feature distributions, correlations, and relationships
3. **Missing Value Handling** – Impute or remove rows with missing values
4. **Feature Scaling/Normalization** – Prepare features for machine learning models
5. **Data Preparation** – Create train/test split and save processed data

---

## Expected Outputs

- EDA visualizations and statistical summaries (saved to `results/figures/`)
- Feature correlation analysis
- Processed dataset (saved to `data/processed/`)
- Preprocessing report and decisions (saved to `results/metrics/`)

---

## 1. Environment Setup and Data Loading

In [1]:
# ============================
# Environment & Configuration
# ============================

import os
import sys
import warnings

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams

# ----------------------------
# Warning configuration
# ----------------------------
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# ----------------------------
# Pandas display options
# ----------------------------
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.4f}".format)

# ----------------------------
# Visualization defaults
# ----------------------------
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")

rcParams["figure.figsize"] = (12, 6)
rcParams["font.size"] = 12

%matplotlib inline

# ----------------------------
# Project paths
# ----------------------------
def find_project_root(start_path):
    """Find the project root from either the repository root or the notebooks directory."""
    current_path = os.path.abspath(start_path)
    for _ in range(3):
        expected_items = [
            os.path.join(current_path, "data"),
            os.path.join(current_path, "notebooks"),
            os.path.join(current_path, "README.md"),
        ]
        if all(os.path.exists(path) for path in expected_items):
            return current_path
        current_path = os.path.dirname(current_path)
    raise FileNotFoundError("Could not locate the project root directory.")

PROJECT_ROOT = find_project_root(os.getcwd())

DATA_RAW_PATH = os.path.join(PROJECT_ROOT, "data", "raw")
DATA_PROCESSED_PATH = os.path.join(PROJECT_ROOT, "data", "processed")
RESULTS_PATH = os.path.join(PROJECT_ROOT, "results")

print("=" * 60)
print("Environment initialized successfully")
print("=" * 60)
print(f"Python       : {sys.version.split()[0]}")
print(f"Numpy        : {np.__version__}")
print(f"Pandas       : {pd.__version__}")
print(f"Scikit-learn : {__import__('sklearn').__version__}")
print("-" * 60)
print(f"Project root : {PROJECT_ROOT}")
print(f"Raw data dir : {DATA_RAW_PATH}")
print("=" * 60)

Environment initialized successfully
Python       : 3.10.19
Numpy        : 2.2.5
Pandas       : 2.3.3
Scikit-learn : 1.7.2
------------------------------------------------------------
Project root : c:\Users\KORISNIK\Documents\drug-discovery-virtual-screening
Raw data dir : c:\Users\KORISNIK\Documents\drug-discovery-virtual-screening\data\raw


### Load raw data from notebook 00

## 2. Exploratory Data Analysis (EDA)

Now we examine the raw dataset to understand its structure, feature distributions, and relationships.
This includes basic statistics and identifying data quality issues.

In [2]:
# Load the raw dataset
dataset_path = os.path.join(DATA_RAW_PATH, "drug_discovery_virtual_screening.csv")

print("Loading raw dataset...")
df = pd.read_csv(dataset_path)

print(f"✓ Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nDataset shape: {df.shape}")
print(f"\nFirst few rows:")
display(df.head())

print(f"\nData types:")
print(df.dtypes)

Loading raw dataset...
✓ Dataset loaded: 2000 rows × 17 columns

Dataset shape: (2000, 17)

First few rows:


,compound_id,protein_id,molecular_weight,logp,h_bond_donors,h_bond_acceptors,rotatable_bonds,polar_surface_area,compound_clogp,protein_length,protein_pi,hydrophobicity,binding_site_size,mw_ratio,logp_pi_interaction,binding_affinity,active
0,CID_00000,PID_361,499.6714,2.4872,1,7,4,113.3508,4.0507,678,6.0197,0.8125,12.5122,0.7370,14.9723,5.9967,0
1,CID_00001,PID_165,436.1736,3.2832,3,4,4,71.9811,3.7044,876,6.4474,0.6514,11.5384,0.4979,21.1683,6.4457,0
2,CID_00002,PID_168,514.7689,NaN,2,11,11,83.9363,1.8696,658,3.9258,0.6335,13.1557,0.7823,9.0741,5.6896,0
3,CID_00003,PID_226,602.3030,3.0381,0,5,5,79.8681,2.4519,312,7.5971,0.5130,12.0718,1.9305,23.0803,6.0434,0
4,CID_00004,PID_224,426.5847,0.6596,2,4,5,88.1987,1.7719,1418,4.2495,0.6136,15.8504,0.3008,2.8028,4.8451,0



Data types:
compound_id             object
protein_id              object
molecular_weight       float64
logp                   float64
h_bond_donors            int64
h_bond_acceptors         int64
rotatable_bonds          int64
polar_surface_area     float64
compound_clogp         float64
protein_length           int64
protein_pi             float64
hydrophobicity         float64
binding_site_size      float64
mw_ratio               float64
logp_pi_interaction    float64
binding_affinity       float64
active                   int64
dtype: object


## 3. Missing Value Handling

Notebook 00 identified three columns with missing values: `logp`, `hydrophobicity`, and `polar_surface_area`.
Each affected column has 60 missing values, corresponding to 3% of the full dataset.

Because the missing values do not occur in exactly the same rows, dropping rows with any missing value removes
174 rows in total, or 8.7% of the original dataset. This is still acceptable for this project because the dataset
remains large enough for train/validation/test splitting and the class distribution stays stable.

The affected features are closely related physicochemical descriptors, and missing values likely reflect incomplete
molecular property computation rather than random measurement noise. Dropping these rows preserves chemical
validity and avoids introducing artificial values through imputation.


In [4]:
print("=" * 60)
print("MISSING VALUE HANDLING")
print("=" * 60)

# Identify missing values
missing_count = df.isna().sum()
missing_pct = (df.isna().mean() * 100)

missing_report = pd.DataFrame({
    "column": missing_count.index,
    "missing_count": missing_count.values,
    "missing_pct": missing_pct.values
}).sort_values("missing_count", ascending=False)

missing_nonzero = missing_report[missing_report["missing_count"] > 0]

print(f"\nColumns with missing values:")
display(missing_nonzero)

print(f"\nStrategy: Drop rows with missing values (only 3% of data affected)")
df_clean = df.dropna()
print(f"Rows before: {len(df)}")
print(f"Rows after:  {len(df_clean)}")
print(f"Rows removed: {len(df) - len(df_clean)}")

# Verify no missing values remain
print(f"\nMissing values after cleaning: {df_clean.isna().sum().sum()}")

MISSING VALUE HANDLING

Columns with missing values:


,column,missing_count,missing_pct
3,logp,60,3.0000
11,hydrophobicity,60,3.0000
7,polar_surface_area,60,3.0000



Strategy: Drop rows with missing values (only 3% of data affected)
Rows before: 2000
Rows after:  1826
Rows removed: 174

Missing values after cleaning: 0


## 4. Feature Analysis and Scaling

We now prepare the cleaned data for machine learning by:
1. Separating features from the target variable
2. Identifying and excluding non-predictive columns (IDs)
3. Scaling all features to zero mean and unit variance using StandardScaler

Feature scaling is essential for:
- Gradient-based models (logistic regression, neural networks)
- Distance-based models (KNN, SVM with RBF kernel)
- Fair feature importance across different value ranges
- Faster convergence during model training

In [5]:
print("=" * 60)
print("FEATURE ANALYSIS AND SCALING")
print("=" * 60)

# Separate features and target
TARGET_COL = "active"
ID_COLS = ["compound_id", "protein_id"]

# Features: all columns except target and IDs
feature_cols = [col for col in df_clean.columns if col not in [TARGET_COL] + ID_COLS]

X = df_clean[feature_cols]
y = df_clean[TARGET_COL].astype(int)

print(f"\nFeature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")
print(f"\nNumber of features: {len(feature_cols)}")

# Verify all features are numeric
print(f"\nData type check:")
non_numeric = X.select_dtypes(exclude=[np.number]).columns
if len(non_numeric) > 0:
    print(f"⚠ Warning: Non-numeric columns found: {list(non_numeric)}")
else:
    print(f"✓ All features are numeric")

print(f"\nTarget distribution:")
print(y.value_counts())
print(f"\nTarget proportions:")
print(y.value_counts(normalize=True))

# Feature scaling
print("\n" + "-" * 60)
print("Scaling features using StandardScaler...")
print("-" * 60)

scaler = StandardScaler()
X_scaled_array = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled_array, columns=feature_cols, index=X.index)

print(f"✓ Features scaled successfully")
print(f"\nScaled data statistics (should be mean≈0, std≈1):")
display(X_scaled.describe())

FEATURE ANALYSIS AND SCALING

Feature matrix shape: (1826, 14)
Target vector shape: (1826,)

Number of features: 14

Data type check:
✓ All features are numeric

Target distribution:
active
0    1270
1     556
Name: count, dtype: int64

Target proportions:
active
0   0.6955
1   0.3045
Name: proportion, dtype: float64

------------------------------------------------------------
Scaling features using StandardScaler...
------------------------------------------------------------
✓ Features scaled successfully

Scaled data statistics (should be mean≈0, std≈1):


,molecular_weight,logp,h_bond_donors,h_bond_acceptors,rotatable_bonds,polar_surface_area,compound_clogp,protein_length,protein_pi,hydrophobicity,binding_site_size,mw_ratio,logp_pi_interaction,binding_affinity
count,1826.0000,1826.0000,1826.0000,1826.0000,1826.0000,1826.0000,1826.0000,1826.0000,1826.0000,1826.0000,1826.0000,1826.0000,1826.0000,1826.0000
mean,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,-0.0000,-0.0000,-0.0000,-0.0000,-0.0000,0.0000,0.0000
std,1.0003,1.0003,1.0003,1.0003,1.0003,1.0003,1.0003,1.0003,1.0003,1.0003,1.0003,1.0003,1.0003,1.0003
min,-3.8812,-4.7481,-1.4512,-2.1938,-2.4827,-4.1130,-3.5667,-1.6954,-3.1748,-3.3057,-3.4877,-1.2953,-2.6614,-3.7632
25%,-0.6533,-0.6439,-0.7127,-0.9068,-0.8122,-0.6705,-0.6798,-0.8976,-0.7078,-0.7096,-0.6686,-0.6879,-0.7145,-0.5621
50%,-0.0119,0.0175,0.0259,-0.0489,0.0231,0.0177,-0.0363,-0.0135,0.0203,0.0244,0.0379,-0.3707,-0.0642,-0.0388
75%,0.6296,0.6386,0.7644,0.8091,0.4407,0.6808,0.6836,0.8967,0.6743,0.6852,0.6753,0.3568,0.6428,0.5342
max,5.0958,3.9638,4.4571,4.2410,3.7817,3.1234,3.4353,1.6997,3.1279,3.5196,3.2462,4.5032,4.9637,7.0744
